In [ ]:
"""
=====================================================================
CSET343 - AI in Healthcare
Lab Assignment 4: Logistic Regression on Breast Cancer Wisconsin (Diagnostic) Dataset
=====================================================================
Objective:
Hands-on experience with logistic regression for binary classification.
Tasks covered:
    Task 1 - Data Acquisition and Exploration
    Task 2 - Data Preprocessing
    Task 3 - Model Training and Basic Evaluation
=====================================================================
"""

# ---------------------------------------------------------------------------
# Import Libraries
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score,
)

sns.set(style="whitegrid")

# =====================================================================
# TASK 1: DATA ACQUISITION AND EXPLORATION
# =====================================================================

# ---------------------------------------------------------------------
# 1. Download and load the dataset
# ---------------------------------------------------------------------
# The Breast Cancer Wisconsin (Diagnostic) dataset is bundled with
# scikit-learn, so it can be loaded directly without an internet
# download. It contains 569 samples, 30 numeric features computed from
# digitized images of breast masses, and a binary target
# (0 = malignant, 1 = benign).
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["Outcome"] = data.target  # 0 = malignant, 1 = benign

print("=" * 70)
print("Dataset shape:", df.shape)
print("=" * 70)
print(df.head())

# ---------------------------------------------------------------------
# 2. Compute summary statistics for all features
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("Summary Statistics (mean, std, min, max, quartiles, etc.)")
print("=" * 70)
summary_stats = df.describe().T
summary_stats["median"] = df.median(numeric_only=True)
print(summary_stats)

# ---------------------------------------------------------------------
# 3. Check for missing values, duplicates, or outliers. Handle them.
# ---------------------------------------------------------------------
print("\n" + "=" * 70)
print("Missing values per column:")
print("=" * 70)
print(df.isnull().sum())

# Handle missing values (if any) by median imputation
if df.isnull().sum().sum() > 0:
    df.fillna(df.median(numeric_only=True), inplace=True)
    print("\nMissing values found and imputed with column median.")
else:
    print("\nNo missing values found in the dataset.")

# Check duplicates
num_duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {num_duplicates}")
if num_duplicates > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

# Check outliers using IQR method (report count per feature)
print("\n" + "=" * 70)
print("Outlier detection using IQR method (count of outliers per feature)")
print("=" * 70)
outlier_counts = {}
for col in data.feature_names:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_counts[col] = len(outliers)

outlier_series = pd.Series(outlier_counts).sort_values(ascending=False)
print(outlier_series.head(10))

# Handling strategy: Instead of deleting outliers (which could remove
# genuinely important malignant cases in a medical dataset), we cap
# extreme values using winsorization (clipping to the IQR bounds).
# This preserves sample size while reducing the influence of extreme values.
for col in data.feature_names:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower_bound, upper_bound)

print("\nOutliers handled via winsorization (clipped to 1.5*IQR bounds).")

# ---------------------------------------------------------------------
# 4. Visualize the data
# ---------------------------------------------------------------------

# (a) Histograms for a few key features
key_features = ["mean radius", "mean texture", "mean perimeter",
                 "mean area", "mean smoothness", "mean concavity"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, feature in zip(axes.flatten(), key_features):
    sns.histplot(df[feature], kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Histogram of {feature}")
plt.tight_layout()
plt.savefig("histograms_key_features.png", dpi=150)
plt.show()

# (b) Box plots for the same key features, split by Outcome
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, feature in zip(axes.flatten(), key_features):
    sns.boxplot(x="Outcome", y=feature, data=df, ax=ax, palette="Set2")
    ax.set_title(f"Boxplot of {feature} by Outcome")
plt.tight_layout()
plt.savefig("boxplots_key_features.png", dpi=150)
plt.show()

# (c) Correlation heatmap
plt.figure(figsize=(20, 16))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, cmap="coolwarm", annot=False, linewidths=0.3)
plt.title("Correlation Heatmap of All Features")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.show()

# (d) Class distribution
plt.figure(figsize=(6, 5))
sns.countplot(x="Outcome", data=df, palette="Set1")
plt.title("Class Distribution (0 = Malignant, 1 = Benign)")
plt.xlabel("Outcome")
plt.ylabel("Count")
plt.savefig("class_distribution.png", dpi=150)
plt.show()

class_counts = df["Outcome"].value_counts()
class_pct = df["Outcome"].value_counts(normalize=True) * 100
print("\n" + "=" * 70)
print("Class Distribution:")
print("=" * 70)
print(f"Malignant (0): {class_counts[0]} samples ({class_pct[0]:.2f}%)")
print(f"Benign    (1): {class_counts[1]} samples ({class_pct[1]:.2f}%)")

# ---------------------------------------------------------------------
# 5. Discuss class imbalance
# ---------------------------------------------------------------------
print("""
Discussion - Class Imbalance:
------------------------------------------------------------------
The dataset shows a moderate class imbalance: benign cases (~63%)
outnumber malignant cases (~37%). While not extreme, this imbalance
can still bias a classifier toward predicting the majority (benign)
class, inflating overall accuracy while under-detecting malignant
(cancerous) cases. In a medical diagnosis context, missing a malignant
case (a False Negative) is far more costly than a false alarm (False
Positive), so metrics like Recall and F1-score for the malignant class,
and stratified sampling during train/test split, are essential to
ensure the model is not misleadingly optimistic about its performance.
------------------------------------------------------------------
""")

# =====================================================================
# TASK 2: DATA PREPROCESSING
# =====================================================================

# ---------------------------------------------------------------------
# Encode the target variable (Outcome) as binary
# ---------------------------------------------------------------------
# Already binary: 0 = malignant, 1 = benign (no further encoding needed).
print("Target variable 'Outcome' is already binary encoded (0/1).")

X = df[data.feature_names]
y = df["Outcome"]

# ---------------------------------------------------------------------
# Split into training and testing sets (80/20) using stratified sampling
# ---------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"\nTraining set size: {X_train.shape[0]} samples")
print(f"Testing set size:  {X_test.shape[0]} samples")
print("\nClass balance check (Training set):")
print(y_train.value_counts(normalize=True))
print("\nClass balance check (Testing set):")
print(y_test.value_counts(normalize=True))

# ---------------------------------------------------------------------
# Standardize the features
# ---------------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures standardized using StandardScaler (mean=0, std=1).")

# =====================================================================
# TASK 3: MODEL TRAINING AND BASIC EVALUATION
# =====================================================================

# ---------------------------------------------------------------------
# Train a logistic regression model
# ---------------------------------------------------------------------
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Predictions
y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]  # probability of class 1 (benign)

# ---------------------------------------------------------------------
# Evaluate the model: Accuracy
# ---------------------------------------------------------------------
accuracy = accuracy_score(y_test, y_pred)
print("\n" + "=" * 70)
print(f"Accuracy: {accuracy:.4f}")
print("=" * 70)

# ---------------------------------------------------------------------
# Precision, Recall, and F1-score (for both classes)
# ---------------------------------------------------------------------
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Malignant (0)", "Benign (1)"]))

precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred)
print("Per-class metrics:")
for i, label in enumerate(["Malignant (0)", "Benign (1)"]):
    print(f"  {label}: Precision={precision[i]:.4f}, Recall={recall[i]:.4f}, "
          f"F1-score={f1[i]:.4f}, Support={support[i]}")

# ---------------------------------------------------------------------
# Confusion matrix (visualize it)
# ---------------------------------------------------------------------
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["Malignant (0)", "Benign (1)"])
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap="Blues", ax=ax, values_format="d")
plt.title("Confusion Matrix - Logistic Regression")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print("\nConfusion Matrix:")
print(cm)

# ---------------------------------------------------------------------
# ROC curve and AUC score
# ---------------------------------------------------------------------
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
auc_score = roc_auc_score(y_test, y_pred_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=150)
plt.show()

print(f"\nROC AUC Score: {auc_score:.4f}")

# ---------------------------------------------------------------------
# Interpretation of results
# ---------------------------------------------------------------------
tn, fp, fn, tp = cm.ravel()
print(f"""
=====================================================================
Interpretation of Results (Medical Diagnosis Context)
=====================================================================
- Accuracy: {accuracy:.4f} -> Overall, the model correctly classifies
  about {accuracy*100:.1f}% of test samples. However, accuracy alone
  can be misleading with imbalanced classes, since a model predicting
  only the majority class could still score high.

- Precision (Malignant): {precision[0]:.4f} -> Of all cases predicted
  as malignant, {precision[0]*100:.1f}% were actually malignant. High
  precision here means fewer healthy patients are wrongly alarmed.

- Recall (Malignant): {recall[0]:.4f} -> Of all actual malignant cases,
  the model correctly identified {recall[0]*100:.1f}%. This is the
  MOST CRITICAL metric in cancer diagnosis: a low recall means real
  cancer cases are being missed (False Negatives = {fn}), which can be
  life-threatening if undetected and untreated.

- F1-score (Malignant): {f1[0]:.4f} -> Balances precision and recall,
  giving a single measure of the model's reliability on the minority
  (malignant) class.

- Confusion Matrix breakdown:
    True Negatives (correctly predicted malignant... i.e. correctly
    identified as class 0): {tn}
    False Positives (benign misclassified as malignant): {fp}
    False Negatives (malignant misclassified as benign): {fn}  <-- most dangerous error type
    True Positives (correctly predicted benign): {tp}

- ROC-AUC: {auc_score:.4f} -> Indicates how well the model separates
  the two classes across all classification thresholds. A value close
  to 1.0 shows the model has excellent discriminative power between
  malignant and benign tumors.

Overall, logistic regression performs strongly on this dataset, but in
a real medical deployment, the classification threshold could be
tuned to further minimize False Negatives (missed cancer diagnoses),
even at the cost of some additional False Positives, since the
consequence of missing a malignant tumor is far more severe than a
false alarm that leads to further testing.
=====================================================================
""")